# Help BOBAI — 동결분류기 + KNN 게이팅 (hello-han 캐글 대회 제출) — Colab

IOAI 2024 On-Site **Help BOBAI** 문제(공식 우수풀이 macro-F1 ≈0.79)를, 사설 캐글 대회 [**hello-han**](https://www.kaggle.com/competitions/hello-han) 에서 다운로드·예측·제출합니다.

**기법**: 이미 배포된 **5-way 분류기(`base_classifier.pth`)를 재학습하지 않고**(파라미터 학습 금지), 신규 클래스(5·6)는 **cosine-KNN 게이팅**으로만 추가 — `which+4 if 신규 else 동결예측`.

**데이터**: `train_dataset.pt` (2473,1,769)=768 임베딩+라벨, `test_dataset.pt` (700,1,768), `base_classifier.pth` (Linear 768→5).
**제출**: `submission.csv` (`ID,class`) → 700개 7-way **macro-F1**.

> ⚙️ scikit-learn 만(CPU, 수 초).
> 🔑 **첫 실행 시 Rules 수락**: 403 이면 [대회 규칙](https://www.kaggle.com/competitions/hello-han/rules)에서 'I Understand and Accept' 1회.
> ⚠️ 노트북에 API 토큰이 평문으로 들어 있습니다. 외부 공유/업로드 금지.

## 0. 설치 · 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "scikit-learn", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("준비 완료")

## 1. Kaggle 에서 hello-han 데이터 다운로드
403 이면 대회 Rules 를 한 번 수락(위 안내).

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("hello-han", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료:", sorted(os.listdir(WORK_DIR)))
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:160])
    print("→ 403 이면 https://www.kaggle.com/competitions/hello-han/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. 동결 분류기 + cosine-KNN 게이팅 & 제출 파일 생성
`train_dataset.pt` 의 768 임베딩·라벨로 **게이트 라벨**(0=기존 0~4, 1=클래스5, 2=클래스6)을 만들고 **`KNeighborsClassifier(n_neighbors=1, weights='distance', metric='cosine')`** 학습. 추론은 게이트가 신규(>0)면 `which+4`(→5·6), 아니면 **동결 5-way base**(`base_classifier.pth`) 예측 → `submission.csv`(`ID,class`) 저장.

In [ ]:
##########################
# Your code here
##########################

## 3. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [hello-han 제출 페이지](https://www.kaggle.com/competitions/hello-han/submit) 에 업로드.

Help BOBAI 모범답안2(동결 분류기 + cosine-KNN 게이팅)를 그대로 옮긴 버전. 핵심은 *배포 모델을 재학습하지 않고 거리(KNN)로만 신규 클래스를 추가*. 로컬 검증 기준 test **macro-F1 ≈0.79**(base-only ≈0.50).